# Toy models of superposition

### What is superposition?

Roughly, the idea of superposition is that neural networks "want to represent more features than they have neurons", so they exploit a property of high-dimensional spaces to simulate a model with many more neurons.

1. Superposition is when a model learns to represent more features than the number of available dimensions in activation space.
2. It does so by representing the features along directions which are a Linear combination of basis. 

Note -
- Set of (>n) directions are called over-complete basis. They are not called basis because they are not linearly independent. 
- [Sparse-coding](http://ufldl.stanford.edu/tutorial/unsupervised/SparseCoding/) is a field of maths that studies techniques to find over-complete basis for a set of vectors such that each vector is a Sparse Linear combination of basis vectors.



<img src="images/superposition_intro.png">

### Key takeaways

- Almost orthogonal vectors: 
Although it's only possible to have n orthogonal vectors in an $n$-dimensional space, it's possible to have $exp(n)$ many "almost orthogonal" ( < $\epsilon$ cosine-similarity) vectors in high-dimensional spaces. See the Johnson–Lindenstrauss lemma.

- Compressed sensing:
In general, if one projects a vector into a lower-dimensional space, one can't reconstruct the original vector. However, this changes if one knows that the original vector is sparse. In this case, it is often possible to recover the original vector.

## How it works -

Features are represented as almost-orthogonal directions in the vector space of neuron outputs. Since the features are only almost-orthogonal, one feature activating looks like other features slightly activating. Tolerating this "noise" or "interference" comes at a cost. But for neural networks with highly sparse features, this cost may be outweighed by the benefit of being able to represent more features! (Crucially, sparsity greatly reduces the costs since sparse features are rarely active to interfere with each other, and non-linear activation functions create opportunities to filter out small amounts of noise.)

<img src="images/superposition_how.png">

## Toy model of Superposition setup

The goal here is that we want to first take input features from high dimensional vector spaces $F$, pass them into our toy model that projects them into an hidden space of lower dimension and then reconstructs the input features back sucessfully. In doing so, the model will learn to have superposition of input features.

Lets consider a simple neural network where we have following architecture - 
- Hidden layer weight matrix that converts Vector space of Input features $F$ to hidden space $H$. 

    i.e $\boxed{f: F \rarr H}$
- Reconstruction weight matrix that reconstructs the input features by converting hidden vectors to output vectors. 

    i.e $\boxed{g: H \rarr F}$

### Dimensions - 

$dim(x) = batch \times d_{inst} \times d_{features}$

$dim(h) = batch \times d_{inst} \times d_{hidden}$

$dim(W) =  d_{inst} \times d_{features} \times d_{hidden}$


### Hidden layer vectors: 

$h = W x$

### Reconstruction of input features: 

$x^\prime = Relu(W^T h + b)$

$\boxed{x^\prime = Relu(W^T Wx + b)}$

### Loss metric: 

$$
\boxed{L = \frac {1}{{BF}} \sum_x \sum_i I_i (x_i - x^\prime_i)^2}
$$

where,

$B$ is batch size

$F$ are number of feautures in the input

$\sum_{x}$ is the summation over elements in batch

$\sum_{i}$ is the summation over each feature

$I_i$ is the importance of i-th feature

$x_i$ is the i-th feature

$x^\prime_i$ is the i-th reconstructed feature

In [1]:
from dataclasses import dataclass
import torch
from torch import nn
import torch.nn.functional as F
from torch import Tensor, tensor
from jaxtyping import Float, Int
import einops
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.graph_objects as go
import plotly.colors as pc
from typing import Callable
from tqdm.notebook import tqdm
import scipy
from scipy.spatial import ConvexHull

device = "mps" if torch.mps.is_available() else "cpu"
torch.set_default_device(device=device)

In [2]:
@dataclass
class ToyModelConfig:
    n_inst: int = 5
    d_feat: int = 5
    d_hidden: int = 2
    device: str = device

## ToyModel class

Here we intend to study the weights and hidden activations of a toy model in non-priviledged basis. Non-priviledged basis are elegant and symmetric, which makes them easy to study. 

A non-priviledged basis is the basis which has no inherent prefered direction. As an example, we can rotate the hidden activations arbitrarily and, as long as we rotate all the weights, have the exact same model behavior.

To illustrate the point, assume we have a orthogonal rotation matrix $R$ (i.e $R^T R = I$) which is used for rotating hidden activations, then,

$h = R W x$

$x^\prime = Relu(R W)^T h + b)$

$x^\prime = Relu((W^T R^T) (R W) + b)$

$x^\prime = Relu(W^T (R^T R) W + b)$

$\boxed{x^\prime = Relu(W^T Wx + b)}$

which leaves the equation unchanged, making our analysis unaffected regardless of the choice of basis.

In [3]:
def constant_lr(*_):
    return 1.0

class ToyModel(nn.Module):
    def __init__(
        self, 
        cfg: ToyModelConfig, 
        importances: float | Tensor = 1.0,
        feature_probabilities: float | Tensor = 0.01,
    ):
        super().__init__()
        self.W = nn.Parameter(torch.empty((cfg.n_inst, cfg.d_feat, cfg.d_hidden), device=cfg.device))
        self.b_final = nn.Parameter(torch.zeros((cfg.n_inst, cfg.d_feat), device=cfg.device))
        nn.init.kaiming_normal_(self.W)
        self.importances = importances.broadcast_to((cfg.n_inst, cfg.d_feat))
        self.feature_probabilities = feature_probabilities.broadcast_to((cfg.n_inst, cfg.d_feat))
        self.cfg = cfg
    
    def forward(self, x: Float[Tensor, "batch inst d_feat"]) -> Float[Tensor, "batch inst d_feat"]:
        h = einops.einsum(self.W, x, "n_inst d_feat d_hidden, batch n_inst d_feat -> batch n_inst d_hidden")
        return F.relu(
            einops.einsum(
                self.W,
                h,
                "n_inst d_feat d_hidden, batch n_inst d_hidden -> batch n_inst d_feat"
            ) + self.b_final
        )
    
    def generate_batch(self, batch: int) -> Float[Tensor, "batch inst d_feat"]:
        rand_seeds = torch.rand((batch, self.cfg.n_inst, self.cfg.d_feat), device=self.cfg.device)
        rand_data = torch.rand((batch, self.cfg.n_inst, self.cfg.d_feat), device=self.cfg.device)
        
        return torch.where(
            rand_seeds <= self.feature_probabilities,
            rand_data,
            0.
        )

    def calculate_loss(
        self, 
        out: Float[Tensor, "batch inst d_feat"], 
        batch: Float[Tensor, "batch inst d_feat"],
    ) -> Float[Tensor, ""]:
        loss = self.importances * ((out - batch) ** 2)
        loss = einops.reduce(loss, "batch inst d_feat -> inst", "mean")

        return loss.sum()
        
    def optimize(
        self,
        batch_size: int = 1024,
        steps: int = 5000,
        log_freq: int = 50,
        lr: float = 1e-3,
        lr_scale: Callable[[int, int], float] = constant_lr,
    ) -> Float[Tensor, ""]:
        optim = torch.optim.AdamW(self.parameters(), lr=lr)
        progress_bar = tqdm(range(steps))

        for step in progress_bar:
            step_lr = lr * lr_scale(step, steps)
            for param_group in optim.param_groups:
                param_group["lr"] = step_lr
            
            optim.zero_grad()
            batch = self.generate_batch(batch_size)
            out = self(batch)
            loss = self.calculate_loss(out, batch)
            loss.backward()
            optim.step()

            if step % log_freq == 0 or step == steps - 1:
                progress_bar.set_postfix(loss=loss.item() / self.cfg.n_inst, lr=step_lr)

## Initialize model config, Feature importances and probabilities

In [4]:
cfg = ToyModelConfig(n_inst=10, d_feat=5, d_hidden=2)

importances = 0.9 ** torch.arange(cfg.d_feat)
feature_probabilities = 50 ** -torch.linspace(0, 1, cfg.n_inst)

## Importances and Feature probabilities

The feature importances and probabilities are defined by the following relationship: 
    
$I^{-i}$, where $i = [0, d_{feat}-1]$

$P^{-j}$, where $j = [0, n_{inst}-1]$


In [5]:
fig = make_subplots(rows=2, cols=1)

fig.add_trace(
    go.Scatter(
        x=list(range(len(importances))),
        y=importances.detach().cpu().numpy(),
        mode='lines+markers',
        name="Importance"
    ),
    row=1, col=1
)
fig.add_trace(
    go.Scatter(
        x=list(range(len(feature_probabilities))),
        y=feature_probabilities.detach().cpu().numpy(),
        mode="lines+markers",
        name="Feature probabilities"
    ),
    row=2, col=1
)
fig.update_xaxes(title_text="Features", row=1, col=1)
fig.update_yaxes(title_text="Importances", row=1, col=1)
fig.update_xaxes(title_text="Instances", row=2, col=1)
fig.update_yaxes(title_text="Probs", row=2, col=1)

fig.show()

In [6]:
model = ToyModel(cfg, importances[None, :], feature_probabilities[:, None]).to(device)
model.optimize(steps=5000)

  0%|          | 0/5000 [00:00<?, ?it/s]

## Analyzing superposition


### Let's first visualise the hidden layer weight matrix

In [7]:
W_dot = einops.einsum(model.W, model.W, "n_inst d_feat1 d_hidden, n_inst d_feat2 d_hidden -> n_inst d_feat1 d_feat2")
W_dot = einops.reduce(W_dot, "n_inst d_feat1 d_feat2 -> d_feat1 d_feat2", "mean")
px.imshow(
    W_dot.detach().cpu().numpy(),
    color_continuous_scale="RdBu_r",
    title="W^T.W",
    color_continuous_midpoint=0,
).show()

### Visualize hidden layer weight vectors in 2D

We observe, that as the sparsity increases, the hidden layer leverages superposition to represent input features of higher dimensions, 5 dimensions in this case. The hidden layer represents 5 dimensional features as verticies of a pentagon, because that minimises the interference between them.

For lower sparsity, 2 most important features are represented as orthogonal vectors in 2-Dimensions.
As sparsity increases, 

In [8]:
def plot_embedding_weights_2D(model: ToyModel):
    W = model.W.detach().cpu()
    feature_importances = model.importances.detach().cpu()
    feature_probabilities = model.feature_probabilities.detach().cpu()
    n_cols = model.cfg.n_inst // 2 + int(model.cfg.n_inst % 2 > 0)
    
    # Normalize importances to [0, 1] for color mapping
    imp_min, imp_max = feature_importances.min(), feature_importances.max()
    norm_importances = (feature_importances - imp_min) / (imp_max - imp_min)
    
    # Get RdBu colorscale
    def get_color(importance_val):
        # Map importance to color (reversed so high importance = red)
        return pc.sample_colorscale(pc.get_colorscale('Viridis'), [importance_val.item()])[0]
    
    fig = make_subplots(
        rows=2, 
        cols=n_cols, 
        subplot_titles=[f"Sparsity: {(1 - feature_probabilities[i][0]).item():.2f}" for i in range(model.cfg.n_inst)],
    )
    for inst in range(model.cfg.n_inst):
        row = inst // n_cols + 1
        col = inst % n_cols + 1
        
        # Add quiver plot (arrows from origin)
        for i in range(model.cfg.d_feat):
            color = get_color(norm_importances[inst, i])
            fig.add_trace(
                go.Scatter(
                    x=[0, W[inst, i, 0].item()],
                    y=[0, W[inst, i, 1].item()],
                    mode='lines+markers+text',
                    line=dict(width=2, color=color),
                    marker=dict(size=[0, 15], symbol=['circle', 'arrow'], angleref='previous', color=color),
                    text=['', f'f_{i}'],
                    textposition='top center',
                    showlegend=False,
                ),
                row=row, col=col
            )
    
    fig.update_xaxes(zeroline=True, zerolinewidth=1, zerolinecolor='lightgray', range=[-1.5, 1.5])
    fig.update_yaxes(zeroline=True, zerolinewidth=1, zerolinecolor='lightgray', range=[-1.5, 1.5])
    
    # Add a dummy trace for colorbar
    fig.add_trace(
        go.Scatter(
            x=[None], y=[None],
            mode='markers',
            marker=dict(
                colorscale='Viridis',
                showscale=True,
                cmin=imp_min.item(),
                cmax=imp_max.item(),
                colorbar=dict(
                    title="Feature Importance",
                    thickness=15,
                    len=0.7,
                    x=1.02
                )
            ),
            hoverinfo='none',
            showlegend=False
        ),
        row=1, col=1
    )
    
    fig.update_layout(
        height=300 * 2, 
        showlegend=False,
        margin=dict(l=40, r=40, t=60, b=40)
    )
    fig.show()

plot_embedding_weights_2D(model)

In the case of hidden layers with higher dimensions than 3. We need a different approach to visualise and analyze the degree of superposition.

## Visualizing 8-dim feature vectors in 3D

For this case we observe that the model learns to arrange 8-dim feature vectors as [Square antiprism](https://en.wikipedia.org/wiki/Square_antiprism) polytopes in hidden layer of 3-dimension.

In [9]:
def plot_embedding_weights_3D(model: ToyModel):
    W = model.W.detach().cpu()
    feature_importances = model.importances.detach().cpu()
    feature_probabilities = model.feature_probabilities.detach().cpu()
    n_cols = model.cfg.n_inst // 2 + int(model.cfg.n_inst % 2 > 0)
    
    # Normalize importances to [0, 1] for color mapping
    imp_min, imp_max = feature_importances.min(), feature_importances.max()
    norm_importances = (feature_importances - imp_min) / (imp_max - imp_min)
    
    # Get RdBu colorscale
    def get_color(importance_val):
        # Map importance to color (reversed so high importance = red)
        return pc.sample_colorscale(pc.get_colorscale('Viridis'), [importance_val.item()])[0]
    
    fig = make_subplots(
        rows=2, 
        cols=n_cols, 
        specs=[[{'type': 'scatter3d'}]*n_cols]*2,
        subplot_titles=[f"Sparsity: {(1 - feature_probabilities[i][0]).item():.2f}" for i in range(model.cfg.n_inst)],
    )
    for inst in range(model.cfg.n_inst):
        row = inst // n_cols + 1
        col = inst % n_cols + 1
        
        # Collect endpoints for surface
        endpoints_x = []
        endpoints_y = []
        endpoints_z = []
        
        # Add quiver plot (arrows from origin)
        for i in range(model.cfg.d_feat):
            color = get_color(norm_importances[inst, i])
            endpoint_x = W[inst, i, 0].item()
            endpoint_y = W[inst, i, 1].item()
            endpoint_z = W[inst, i, 2].item()
            
            endpoints_x.append(endpoint_x)
            endpoints_y.append(endpoint_y)
            endpoints_z.append(endpoint_z)
            
            fig.add_trace(
                go.Scatter3d(
                    x=[0, endpoint_x],
                    y=[0, endpoint_y],
                    z=[0, endpoint_z],
                    mode='lines+markers+text',
                    line=dict(width=2, color=color),
                    marker=dict(
                        size=[0, model.cfg.d_feat], 
                        # symbol=['circle', 'arrow'], 
                        # angleref='previous', 
                        color=color,
                    ),
                    text=['', f'f_{i}'],
                    textposition='top center',
                    showlegend=False,
                ),
                row=row, col=col
            )
        
        # Add surface connecting the endpoints
        # Create convex hull surface
        if len(endpoints_x) >= 4:  # Need at least 4 points for a 3D surface
            points = torch.tensor([[x, y, z] for x, y, z in zip(endpoints_x, endpoints_y, endpoints_z)])
            try:
                hull = ConvexHull(points.cpu().numpy())
                fig.add_trace(
                    go.Mesh3d(
                        x=points[:, 0].cpu().numpy(),
                        y=points[:, 1].cpu().numpy(),
                        z=points[:, 2].cpu().numpy(),
                        i=hull.simplices[:, 0],
                        j=hull.simplices[:, 1],
                        k=hull.simplices[:, 2],
                        opacity=0.2,
                        color='lightblue',
                        showlegend=False,
                    ),
                    row=row, col=col
                )
            except Exception as e:
                print(f"Convex hull failed for instance {inst}: {e}")
                pass  # Skip if convex hull fails
    
    fig.update_layout(
        height=300 * 2, 
        showlegend=False,
        margin=dict(l=40, r=40, t=60, b=40)
    )
    fig.show()

cfg = ToyModelConfig(n_inst=10, d_feat=8, d_hidden=3)

importances = 0.9 ** torch.arange(cfg.d_feat)
feature_probabilities = 50 ** -torch.linspace(0, 1, cfg.n_inst)

model = ToyModel(cfg, importances[None, :], feature_probabilities[:, None]).to(device)
model.optimize(steps=5000)

plot_embedding_weights_3D(model)

  0%|          | 0/5000 [00:00<?, ?it/s]

### Computing degree of superposition:

We can analyse the degree of superposition in the hidden layer by following metric - 

$\sum_{j} (\hat{W_i} \cdot W_j)^2$

Explanation - 

Given i-th feature in the hidden dimension. We want to see, what is the magnitude of projection from other features on it. The projection from all other features can be squared and added to make sure we don't cancel out +ve projections with -ve ones. We also want to ensure that we don't compute self-projections. 

While computing the projections, we build a tensor called `interference`, which has a shape - `[n_inst, d_feat1, d_feat2]`. 

where, `d_feat1` represents the feature direction along which `d_feat2` projects.

In [10]:
cfg = ToyModelConfig(n_inst=10, d_feat=80, d_hidden=20)

importances = 0.9 ** torch.arange(cfg.d_feat)
feature_probabilities = 50 ** -torch.linspace(0, 1, cfg.n_inst)

model = ToyModel(cfg, importances[None, :], feature_probabilities[:, None]).to(device)
model.optimize(steps=10_000)

  0%|          | 0/10000 [00:00<?, ?it/s]

In [11]:
def plot_superposition_score(model: ToyModel) -> float:
    feature_probabilities = model.feature_probabilities.detach().cpu()
    W = model.W.detach().cpu()

    W_norm = W / (1e-5 + torch.norm(W, p=2, dim=-1, keepdim=True))

    # Tensor containing the interference between features for each instance
    # d_feat1 is the feature direction on which d_feat2 projects
    # this is done for each instance separately with varying sparsity.
    interference = einops.einsum(
        W_norm, 
        W, 
        "n_inst d_feat1 d_hidden, n_inst d_feat2 d_hidden -> n_inst d_feat1 d_feat2",
    )
    indices = torch.arange(model.cfg.d_feat, device="cpu")
    interference[:, indices, indices] = 0.

    # Once we have the interference matrix, we can compute the polysemanticity score
    # We just take the norm of the interference to get a magintude.
    polysemanticity = interference.norm(dim=-1)
    
    WtW = einops.einsum(W, W, "n_inst d_feat d_hidden, n_inst d_feat2 d_hidden -> n_inst d_feat d_feat2")
    norms = W.norm(p=2, dim=-1)

    for inst in range(model.cfg.n_inst):
        fig = make_subplots(
            rows=1, cols=2,
            subplot_titles=("Feature Norms", "W^T·W")
        )
        fig.add_trace(
            go.Bar(
                x=indices.numpy(),
                y=norms[inst].numpy(),
                name="||Wi||",
                marker=dict(
                    color=polysemanticity[inst].numpy(),
                    cmin=0,
                    cmax=1,
                )
            ),
            row=1, col=1
        )
        fig.add_trace(
            go.Heatmap(
                z=WtW[inst].numpy(),
                colorscale="RdBu_r",
                zmid=0,
            ),
            row=1, col=2
        )
        
        fig.update_xaxes(title_text="Features", row=1, col=1)
        fig.update_yaxes(title_text="||Wi||", row=1, col=1)
        fig.update_xaxes(title_text="Feature i", row=1, col=2)
        fig.update_yaxes(title_text="Feature j", row=1, col=2, autorange="reversed")
        fig.update_layout(title_text=f"Sparsity: {(1 - feature_probabilities[inst][0]).item():.2f}", showlegend=False, width=700, height=400)
        fig.show()


plot_superposition_score(model)


## Interpretation of the plots

- For instances with low sparsity, we see no projects under `Feature norms` graph (represented by dark blue lines) and orthogonal features under $W^T \cdot W$ graph (represented by identity matrix). 
    
    Since most of the features activate simultaneously, the model fails to project all the input features onto a lower dimensional hidden layer, so the superposition is close `0`. The model forced to represent only `d_hidden` features out of `d_feat`, which it picks arbitrarily. This is shown as dark blue lines in `Feature norms` graph and as dark red identity matrix for $W^T W$ graph.

- For instances with medium sparsity, we see projections occuring leading to higher level of superpositions. 

    One interesting observation is that the model learns to split input feature space into smaller feature geometries. For example, in some cases, 5-D feature space is split into 2-D + 3-D feature space.

## Priviledged basis model

Next we study a simplest toy model with priviledged basis. This can be achieved by considering a network with following configuration - 

$h = Relu(W x)$

$x^\prime = Relu(W^T h + b)$

By having such setup, the model is forced to chose a basis for computation as it can no longer have an arbitrary orthogonal matrix (like a rotation matrix $R$ that we discussed before).

In [12]:
class NeuronModel(ToyModel):
    def forward(self, features: Float[Tensor, "batch inst d_feat"]):
        h = F.relu(einops.einsum(
            self.W,
            features,
            "n_inst d_feat d_hidden, batch n_inst d_feat -> batch n_inst d_hidden"
        ))

        return F.relu(
            einops.einsum(
                self.W,
                h,
                "n_inst d_feat d_hidden, batch n_inst d_hidden -> batch n_inst d_feat"
            ) + self.b_final
        )

cfg = ToyModelConfig(n_inst=7, d_feat=10, d_hidden=5)

importances = 0.9 ** torch.arange(1, 1+ cfg.d_feat)
feature_probabilities = torch.tensor([0.75, 0.35, 0.15, 0.1, 0.06, 0.02, 0.01])

model = NeuronModel(cfg, importances[None, :], feature_probabilities[:, None]).to(device)
model.optimize(steps=10_000)

  0%|          | 0/10000 [00:00<?, ?it/s]

In [13]:
plot_superposition_score(model)

### All these experiments confirm the superposition hypothesis!